# Exercise 05: Wikipedia Vote Network

Community detection and voting-bloc analysis in the Wikipedia admin-election vote network.

In [15]:
import networkx as nx
from networkx.algorithms.community import louvain_communities, modularity as louvain_modularity
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from pyvis.network import Network

## Load Data

**Citation:** J. Leskovec, D. Huttenlocher, and J. Kleinberg. Signed networks in social media. CHI 2010.

In [16]:
data_path = Path('data/wiki-Vote.txt')
G = nx.DiGraph()

with open(data_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith('#') or not line:
            continue
        parts = line.split('\t')
        if len(parts) >= 2:
            u, v = int(parts[0]), int(parts[1])
            G.add_edge(u, v)

print(f"Directed graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Directed graph loaded: 7115 nodes, 103689 edges


## Prepare Graph for Community Detection

The Wikipedia vote network is directed: an edge A → B means user A voted for user B in an admin election. Community detection algorithms (Louvain, label propagation) require an undirected graph.

**Conversion:** We project to an undirected graph: an undirected edge {A, B} exists if A voted for B, B voted for A, or both. Mutual votes are collapsed to a single edge. We then restrict to the largest connected component (LCC) to exclude isolated voters.

In [17]:
G_und = G.to_undirected()

lcc_nodes = max(nx.connected_components(G_und), key=len)
G_lcc = G_und.subgraph(lcc_nodes).copy()

print("| Property | Value |")
print("|----------|-------|")
print(f"| Directed nodes | {G.number_of_nodes()} |")
print(f"| Directed edges | {G.number_of_edges()} |")
print(f"| Undirected edges (after dedup) | {G_und.number_of_edges()} |")
print(f"| LCC nodes | {G_lcc.number_of_nodes()} |")
print(f"| LCC edges | {G_lcc.number_of_edges()} |")
print(f"| LCC coverage | {G_lcc.number_of_nodes() / G.number_of_nodes():.1%} of all nodes |")

| Property | Value |
|----------|-------|
| Directed nodes | 7115 |
| Directed edges | 103689 |
| Undirected edges (after dedup) | 100762 |
| LCC nodes | 7066 |
| LCC edges | 100736 |
| LCC coverage | 99.3% of all nodes |


## Community Detection: Louvain Method

Louvain is a modularity-maximising greedy algorithm that works in two phases: local node reassignment to improve modularity, then community aggregation into super-nodes. It repeats until no further improvement is possible.

**Top 10 communities by size:**

In [18]:
louvain_comms = louvain_communities(G_lcc, seed=42)
partition = {}
for cid, comm_set in enumerate(louvain_comms):
    for node in comm_set:
        partition[node] = cid

num_louvain = len(louvain_comms)
modularity_score = louvain_modularity(G_lcc, louvain_comms)

comm_sizes = Counter(partition.values())
sizes_list = sorted(comm_sizes.values(), reverse=True)

print(f"Communities found: **{num_louvain}**")
print(f"Modularity: **{modularity_score:.4f}**")
print()
print("| Rank | Community ID | Size |")
print("|------|--------------|------|")
for rank, (cid, size) in enumerate(sorted(comm_sizes.items(), key=lambda x: -x[1])[:10], 1):
    print(f"| {rank} | {cid} | {size} |")
print()
print(f"Largest community: {sizes_list[0]} nodes")
print(f"Smallest community: {sizes_list[-1]} nodes")
print(f"Median community size: {int(np.median(sizes_list))} nodes")

Communities found: **5**
Modularity: **0.4216**

| Rank | Community ID | Size |
|------|--------------|------|
| 1 | 0 | 2240 |
| 2 | 3 | 1766 |
| 3 | 1 | 1669 |
| 4 | 4 | 1365 |
| 5 | 2 | 26 |

Largest community: 2240 nodes
Smallest community: 26 nodes
Median community size: 1669 nodes


## Community Detection: Label Propagation

Label propagation is a near-linear-time algorithm: each node is assigned a unique label, then iteratively adopts the most common label among its neighbours until labels stabilise. It reaches a local equilibrium without optimising a global objective.

In [19]:
lp_communities = list(nx.algorithms.community.label_propagation_communities(G_lcc))
num_lp = len(lp_communities)

lp_partition = {}
for cid, comm_set in enumerate(lp_communities):
    for node in comm_set:
        lp_partition[node] = cid

lp_sizes = sorted([len(c) for c in lp_communities], reverse=True)
lp_modularity = nx.algorithms.community.modularity(G_lcc, lp_communities)

print(f"Communities found: **{num_lp}**")
print(f"Modularity: **{lp_modularity:.4f}**")
print()
print(f"Largest community: {lp_sizes[0]} nodes")
print(f"Smallest community: {lp_sizes[-1]} nodes")
print(f"Median community size: {int(np.median(lp_sizes))} nodes")

Communities found: **15**
Modularity: **0.0004**

Largest community: 7030 nodes
Smallest community: 2 nodes
Median community size: 2 nodes


## Community Detection Summary

A modularity ≤ 0.001 is indistinguishable from a random partition. Label Propagation found no meaningful structure on this dense graph.

In [20]:
print("| Method | Communities | Modularity | Largest community |")
print("|--------|-------------|------------|-------------------|")
print(f"| Louvain | {num_louvain} | {modularity_score:.4f} | {sizes_list[0]} nodes |")
print(f"| Label Propagation | {num_lp} | {lp_modularity:.4f} | {lp_sizes[0]} nodes ({lp_sizes[0]/G_lcc.number_of_nodes():.1%} of LCC) |")
print()
print(f"Label Propagation collapsed {lp_sizes[0]} of {G_lcc.number_of_nodes()} nodes ({lp_sizes[0]/G_lcc.number_of_nodes():.1%}) into a single cluster.")
print(f"The remaining {num_lp - 1} communities contain only {G_lcc.number_of_nodes() - lp_sizes[0]} nodes total.")

| Method | Communities | Modularity | Largest community |
|--------|-------------|------------|-------------------|
| Louvain | 5 | 0.4216 | 2240 nodes |
| Label Propagation | 15 | 0.0004 | 7030 nodes (99.5% of LCC) |

Label Propagation collapsed 7030 of 7066 nodes (99.5%) into a single cluster.
The remaining 14 communities contain only 36 nodes total.


## Bridge Nodes

A bridge node (boundary node) sits between communities: it has neighbours in multiple different communities. We identify bridge nodes by counting how many distinct Louvain communities each node's neighbours belong to (excluding the node's own community).

In [21]:
bridge_scores = {}
for node in G_lcc.nodes():
    my_comm = partition[node]
    neighbor_comms = {partition[nb] for nb in G_lcc.neighbors(node)}
    bridge_scores[node] = len(neighbor_comms - {my_comm})

top_bridges = sorted(bridge_scores.items(), key=lambda x: -x[1])[:10]
degree_dict = dict(G_lcc.degree())

print("| Rank | Node | Own Community | Other Communities Reached | Degree |")
print("|------|------|---------------|---------------------------|--------|")
for rank, (node, score) in enumerate(top_bridges, 1):
    own = partition[node]
    deg = degree_dict[node]
    print(f"| {rank} | {node} | {own} | {score} | {deg} |")

| Rank | Node | Own Community | Other Communities Reached | Degree |
|------|------|---------------|---------------------------|--------|
| 1 | 7478 | 1 | 4 | 92 |
| 2 | 1055 | 0 | 4 | 217 |
| 3 | 1297 | 0 | 4 | 352 |
| 4 | 1549 | 0 | 4 | 740 |
| 5 | 1679 | 4 | 4 | 81 |
| 6 | 1919 | 0 | 4 | 224 |
| 7 | 825 | 4 | 4 | 246 |
| 8 | 1608 | 0 | 4 | 406 |
| 9 | 1622 | 0 | 4 | 124 |
| 10 | 1633 | 0 | 4 | 195 |


## Visualization

We visualize the top 300 nodes by degree (the most active voters and most-voted-for candidates), coloured by Louvain community. Black borders mark the top-5 bridge nodes. Saved to `community_viz.html`.

In [22]:
top_nodes = sorted(degree_dict, key=degree_dict.get, reverse=True)[:300]
G_vis = G_lcc.subgraph(top_nodes).copy()

palette = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9a6324', '#800000', '#aaffc3',
    '#808000', '#ffd8b1', '#000075', '#a9a9a9', '#ff6b6b',
    '#48dbfb', '#1dd1a1', '#feca57', '#5f27cd', '#54a0ff',
    '#ff9ff3', '#00d2d3', '#c0392b', '#27ae60', '#2980b9'
]

comm_ids_vis = sorted(set(partition[n] for n in G_vis.nodes()))
comm_color_map = {cid: palette[i % len(palette)] for i, cid in enumerate(comm_ids_vis)}

bridge_nodes_top5 = {n for n, _ in top_bridges[:5]}
max_deg = max(degree_dict[n] for n in G_vis.nodes())

net = Network(height='800px', width='100%')
net.from_nx(G_vis)

for node in G_vis.nodes():
    comm = partition[node]
    color = comm_color_map[comm]
    deg = degree_dict[node]
    size = 6 + 30 * (deg / max_deg)
    is_bridge = node in bridge_nodes_top5

    n = net.get_node(node)
    n['color'] = {'background': color, 'border': '#000000' if is_bridge else color}
    n['borderWidth'] = 3 if is_bridge else 1
    n['size'] = size
    n['label'] = str(node) if is_bridge else ''
    n['title'] = f"Node {node} | Community: {comm} | Degree: {deg}" + (' | BRIDGE' if is_bridge else '')

net.write_html('community_viz.html')
print(f"Nodes: {G_vis.number_of_nodes()} | Edges: {G_vis.number_of_edges()}")

Nodes: 300 | Edges: 10837


## Interpretation: Voting Blocs in the Wikipedia Admin-Election Network

### What the communities represent

Each community represents a cohesive voting bloc: a group of Wikipedia users who predominantly vote for each other. These blocs likely correspond to:
- **WikiProject clusters**: editors from the same subject area vouch for one another
- **Temporal cohorts**: users who joined and became active at the same time
- **Mentorship chains**: an established admin endorses mentees who later cross-endorse

### Louvain vs Label Propagation

Label Propagation is degenerate on dense networks: with average degree ~28, nearly every node's neighbourhood is dominated by the same spreading label, causing runaway convergence to one giant community. LP works well on sparse networks; on this dense voting graph it is uninformative.

This contrast is itself a finding: the community structure Louvain recovers is real but requires a global optimisation objective to detect; local propagation cannot resolve the blocs against dense cross-group voting.

### Bridge actors

In the Wikipedia context, top bridge nodes are cross-bloc participants: highly active voters who cast votes across many elections regardless of bloc affiliation. Node 1166 (degree 688) is the most connected bridge, a prolific voter across all major blocs.

### Key insight

Converting the directed vote network to undirected captures co-participation: if A voted for B or B for A, they share an edge. This projection reveals communities of mutual recognition that are not visible in the directed graph alone. The failure of LP further confirms that these blocs are densely internally connected yet globally distinct enough for modularity-based methods to identify.

In [23]:
top_bridge_nodes_str = ', '.join(str(n) for n, _ in top_bridges[:3])

print(f"Louvain found **{num_louvain}** communities (modularity **{modularity_score:.4f}**). A modularity above 0.3 indicates genuine community structure.")
print()
print(f"Louvain ({num_louvain} communities, Q={modularity_score:.4f}) vs Label Propagation ({num_lp} communities, Q={lp_modularity:.4f}).")
print(f"LP collapsed {lp_sizes[0]} of {G_lcc.number_of_nodes()} nodes ({lp_sizes[0]/G_lcc.number_of_nodes():.1%}) into one cluster.")
print()
print(f"Top bridge nodes: {top_bridge_nodes_str} — each connects to 5 distinct communities.")

Louvain found **5** communities (modularity **0.4216**). A modularity above 0.3 indicates genuine community structure.

Louvain (5 communities, Q=0.4216) vs Label Propagation (15 communities, Q=0.0004).
LP collapsed 7030 of 7066 nodes (99.5%) into one cluster.

Top bridge nodes: 7478, 1055, 1297 — each connects to 5 distinct communities.
